# A02 — Portfolio Investment Decision (10 Features)

> For Product Owners, Portfolio Owners, and Agile leaders who need a clear view of the portfolio investment decision. Risk Managers will find the risk metrics in the decision-basis sections.

---

**The central question:** Should we pay the full development investment upfront, or spread it over annual installments?

| | Option A — Upfront | Option B — Installment |
|---|---|---|
| **Development cost** | Paid in full at Year 0 | Equal annual payments over *n* years |
| **Operating cost (OpEx)** | Every year | Every year (identical) |
| **Time-value effect** | None — full PV of investment | Favourable — later payments discounted |
| **Year-1 cash outflow** | Maximum | Minimum |

This notebook makes the cash flows explicit — year by year, feature by feature — so the NPV advantage of installment financing is visible, not just asserted.

---

*Previous: [A01 — Portfolio Advisor](01-portfolio-advisor.ipynb)*

In [ ]:
import math

from fhs.application import BlockchainCaseStudyService, CapitalBudgetingContext
from fhs.notebook import notebook_setup
from fhs.presentation.notebook import COLORS, show

setup = notebook_setup("advanced-features")

if setup.scenario is None:
    raise RuntimeError("Scenario setup could not be initialized")

case = BlockchainCaseStudyService(
    seed=setup.scenario.seed, scenarios=setup.scenario.scenarios
)
ctx: CapitalBudgetingContext = case.capital_budgeting_context(setup.scenario, years=3)

show.info(
    f"Scenario loaded: {len(ctx.features_by_key)} features · "
    f"{ctx.scenarios:,} scenarios · discount rate {ctx.discount_rate:.0%}"
)

In [ ]:
def _irr_hurdle_gauge(v: float, hurdle: float) -> str:
    """Mini SVG gauge separating hurdle (above) and IRR (below) markers.

    Layout (44px tall) with a 2px gap between each triangle and the bar:
      hurdle X%        ← y=0..8 label
        ▼              ← y=10..16 triangle (tip at y=16; 2px gap to bar)
      ━━━━━━━━━━━     ← y=18..26 bar
              ▲        ← y=28..34 triangle (tip at y=28; 2px gap from bar)
      0%        max    ← y=42 scale labels

    Hurdle and IRR markers occupy non-overlapping vertical bands AND
    do not touch the bar — they read as separate, free-standing markers.
    """
    scale_max = max(hurdle * 2.0, 0.25)
    hx = 4.0 + (hurdle / scale_max) * 152.0
    ix = 4.0 + (max(v, 0.0) / scale_max) * 152.0
    above = v >= hurdle
    fill_color = COLORS.success_vivid if above else COLORS.danger_vivid
    fill_w = max(ix - 4.0, 0.0)
    return (
        f"<svg width='160' height='44' viewBox='0 0 160 44' "
        f"xmlns='http://www.w3.org/2000/svg' "
        f"style='display:block;margin:6px auto 0'>"
        f"<text x='{hx:.1f}' y='8' font-size='9' font-weight='700' "
        f"fill='{COLORS.neutral}' text-anchor='middle'>"
        f"hurdle {hurdle:.0%}</text>"
        f"<polygon points='{hx - 4:.1f},10 {hx + 4:.1f},10 {hx:.1f},16' "
        f"fill='{COLORS.neutral}'/>"
        f"<rect x='4' y='18' width='152' height='8' rx='4' "
        f"fill='{COLORS.surface}' stroke='{COLORS.border}' stroke-width='1'/>"
        f"<rect x='4' y='18' width='{fill_w:.1f}' height='8' rx='4' "
        f"fill='{fill_color}'/>"
        f"<polygon points='{ix - 5:.1f},34 {ix + 5:.1f},34 {ix:.1f},28' "
        f"fill='{fill_color}'/>"
        f"<text x='4' y='42' font-size='9' fill='{COLORS.neutral}'>0%</text>"
        f"<text x='156' y='42' font-size='9' fill='{COLORS.neutral}' "
        f"text-anchor='end'>{scale_max:.0%}</text>"
        f"</svg>"
    )


def _irr_kpi(v: float) -> tuple[str, str]:
    """Rich IRR KPI: big number + gauge/banner emphasising bad IRR cases.

    Four tiers:
    - NaN  → ∞ + green "positive from Y1" pill
    - ≤ 0  → — + red "NEVER BREAKS EVEN" warning banner (no gauge — value is off-scale)
    - 0 < v < hurdle → red % + red gauge showing IRR below hurdle + verdict
    - v ≥ hurdle    → green % + green pill "X pp above hurdle"
    """
    hurdle = ctx.discount_rate

    if math.isnan(v):
        col = COLORS.success
        pill = (
            f"<span style='display:inline-block;background:{COLORS.success_surface};"
            f"color:{COLORS.success_vivid};border:1.5px solid {COLORS.success_border};"
            f"padding:2px 10px;border-radius:12px;font-size:11px;font-weight:700;"
            f"margin-top:4px'>↑ Positive from Year 1</span>"
        )
        val_html = (
            f"<div style='font-size:28px;font-weight:800;color:{col};"
            f"line-height:1.0'>∞</div>{pill}"
        )
        return val_html, col

    if v <= 0:
        col = COLORS.danger
        banner = (
            f"<div style='background:{COLORS.danger_surface};"
            f"border-left:4px solid {COLORS.danger_vivid};padding:6px 10px;"
            f"margin-top:6px;border-radius:6px;text-align:left;line-height:1.25'>"
            f"<div style='font-size:12px;font-weight:800;"
            f"color:{COLORS.danger_vivid};letter-spacing:0.02em'>"
            f"⊘ NEVER BREAKS EVEN</div>"
            f"<div style='font-size:10px;font-weight:600;"
            f"color:{COLORS.danger};margin-top:2px'>"
            f"NPV stays negative at any positive discount rate</div></div>"
        )
        val_html = (
            f"<div style='font-size:28px;font-weight:800;color:{col};"
            f"line-height:1.0'>—</div>{banner}"
        )
        return val_html, col

    if v >= hurdle:
        col = COLORS.success
        delta_pp = (v - hurdle) * 100.0
        pill = (
            f"<span style='display:inline-block;background:{COLORS.success_surface};"
            f"color:{COLORS.success_vivid};border:1.5px solid {COLORS.success_border};"
            f"padding:2px 10px;border-radius:12px;font-size:11px;font-weight:700;"
            f"margin-top:4px;white-space:nowrap'>"
            f"↑ {delta_pp:.1f} pp above {hurdle:.0%} hurdle</span>"
        )
        val_html = (
            f"<div style='font-size:28px;font-weight:800;color:{col};"
            f"line-height:1.0'>{v:.1%}</div>{pill}"
        )
        return val_html, col

    col = COLORS.danger
    delta_pp = (hurdle - v) * 100.0
    verdict = (
        f"<div style='font-size:11px;font-weight:800;color:{COLORS.danger_vivid};"
        f"line-height:1.3;margin-top:4px;text-align:center'>"
        f"⚠ {delta_pp:.1f} pp below {hurdle:.0%} hurdle</div>"
        f"<div style='font-size:10px;font-weight:600;color:{COLORS.danger};"
        f"line-height:1.2;text-align:center'>"
        f"Yield falls short of cost of capital</div>"
    )
    val_html = (
        f"<div style='font-size:28px;font-weight:800;color:{col};"
        f"line-height:1.0'>{v:.1%}</div>"
        f"{_irr_hurdle_gauge(v, hurdle)}"
        f"{verdict}"
    )
    return val_html, col


npv_benefit = ctx.portfolio_npv_b.expected - ctx.portfolio_npv_a.expected
_irr_a_html, _irr_color_a = _irr_kpi(ctx.portfolio_irr_a.expected)
_irr_b_html, _irr_color_b = _irr_kpi(ctx.portfolio_irr_b.expected)

_executive_items = [
    ("Total Portfolio Investment", f"€{ctx.total_cost:,.0f}", COLORS.neutral),
    ("Annual OpEx (all features)", f"€{ctx.total_annual_opex:,.0f}/yr", COLORS.neutral),
    ("Discount Rate (hurdle)", f"{ctx.discount_rate:.0%}", COLORS.neutral),
    (
        "NPV — Option A (Upfront)",
        f"€{ctx.portfolio_npv_a.expected:,.0f}",
        COLORS.secondary,
    ),
    (
        "NPV — Option B (Installment)",
        f"€{ctx.portfolio_npv_b.expected:,.0f}",
        COLORS.secondary,
    ),
    ("NPV Advantage of Option B", f"+€{npv_benefit:,.0f}", COLORS.success),
    ("IRR — Option A", _irr_a_html, _irr_color_a),
    ("IRR — Option B", _irr_b_html, _irr_color_b),
    ("PI — Option A (Upfront)", f"{ctx.pi_a:.2f}×", COLORS.secondary),
    ("PI — Option B (Installment)", f"{ctx.pi_b:.2f}×", COLORS.success),
    ("Year-1 Cash Outflow — Option A", f"€{ctx.total_cost:,.0f}", COLORS.danger),
    (
        "Year-1 Cash Outflow — Option B",
        f"€{ctx.total_annual_installment:,.0f}",
        COLORS.success,
    ),
]

In [ ]:
show.executive(
    "Portfolio Investment Decision — Executive Summary",
    _executive_items,
    footer=(
        f"NPV floor (VaR 95%) — A: €{ctx.portfolio_npv_a.var_95:,.0f} · "
        f"B: €{ctx.portfolio_npv_b.var_95:,.0f}. "
        f"PI = NPV / PV(investment) — Option B higher because installments are discounted. "
        f"Monte Carlo: {ctx.scenarios:,} scenarios per feature."
    ),
)

---

## 1) Year 1 — What does each feature earn and what does it cost?

The table below is the starting point for both options. It shows expected business value alongside the Year-1 cost exposure under each financing structure.

- **Option A Net Year 1** = Expected BV − Full Investment − OpEx. Negative is expected for multi-year programmes.
- **Option B Net Year 1** = Expected BV − Annual Installment − OpEx. A positive net here means the feature is P&L-neutral from day one.

In [ ]:
show.year1_overview(
    case.year1_overview_rows(ctx.year1, ctx.portfolio_overview),
    title="Year 1 Overview — Business Value vs. Financing Options",
)

In [ ]:
show.feature_cashflow_comparison(
    ctx.feat_sched_a,
    ctx.feat_sched_b,
    title="3-Year Net Cash Flows — All Hypotheses + Portfolio",
    discount_rate=ctx.discount_rate,
    irr_rows=case.irr_dual_rows(ctx.multi_year, ctx.features_by_key),
    portfolio_irr_a=ctx.portfolio_irr_a.expected,
    portfolio_irr_b=ctx.portfolio_irr_b.expected,
)

---

## 2) IRR — Which Hypotheses Clear the Hurdle?

**IRR (Internal Rate of Return)** is the discount rate at which NPV = 0 — the annualised yield the portfolio earns on its invested capital. The decision rule: **IRR above the hurdle rate creates value; below it destroys value.**

The *IRR at a Glance* panel above shows the result for every feature. Compare each feature's IRR against the hurdle rate to see which ones create value and which ones fall short.

### Why Option B yields a higher IRR

Both options start with a negative cashflow at Year 0, so both always produce a finite IRR. The gap comes from the **size of the Year-0 outflow**:

| | Year 0 | Years 1 – (n−1) | Year n onward |
|---|---|---|---|
| **Option A — Upfront** | −full investment | +BV − OpEx | +BV − OpEx |
| **Option B — Installment** | −1st installment | +BV − installment − OpEx | +BV − OpEx |

Option B's Year-0 outflow is a fraction of Option A's. A smaller capital commitment must be recovered for NPV to reach zero, so the break-even discount rate — the IRR — is higher. **Less capital at risk early means a higher yield on that capital.**

The NPV-vs-rate curve below makes this visible: both curves cross zero (IRR), but Option B crosses at a higher rate.

In [ ]:
show.npv_rate_curve(
    ctx.feat_sched_a,
    ctx.feat_sched_b,
    hurdle_rate=ctx.discount_rate,
    irr_a=ctx.portfolio_irr_a.expected
    if not math.isnan(ctx.portfolio_irr_a.expected)
    else None,
    irr_b=ctx.portfolio_irr_b.expected
    if not math.isnan(ctx.portfolio_irr_b.expected)
    else None,
    title=f"NPV vs. Discount Rate — Option A vs. Option B (hurdle {ctx.discount_rate:.0%})",
)

In [ ]:
show.capital_budgeting_summary(
    npv_a=ctx.portfolio_npv_a,
    npv_b=ctx.portfolio_npv_b,
    irr_a=ctx.portfolio_irr_a,
    irr_b=ctx.portfolio_irr_b,
    pi_a=ctx.pi_a,
    pi_b=ctx.pi_b,
    discount_rate=ctx.discount_rate,
    title="Portfolio Investment Summary",
    verdict_message=ctx.npv_a_assessment.primary_message,
    verdict_is_go=(ctx.npv_a_assessment.verdict == "GO"),
)

---

## 3) Visual Comparison

Two charts that make the financing trade-off visible without reading a table.

- **Left chart** — net cash flow year by year: the red bar at Year 0 for Option A is the full investment outflow. Option B has no Year-0 bar.
- **Right chart** — NPV side by side with the VaR 95% floor (red markers) and the Profitability Index above each bar.

In [ ]:
show.cashflow_chart(
    ctx.feat_sched_a,
    ctx.feat_sched_b,
    title="Net Cash Flow per Year — Option A vs. Option B (Portfolio)",
)

In [ ]:
show.npv_comparison_chart(
    ctx.portfolio_npv_a,
    ctx.portfolio_npv_b,
    pi_a=ctx.pi_a,
    pi_b=ctx.pi_b,
    discount_rate=ctx.discount_rate,
    title=f"Portfolio NPV — Option A vs. Option B (hurdle {ctx.discount_rate:.0%})",
)

In [ ]:
rec = case.financing_recommendation(
    npv_a=ctx.portfolio_npv_a,
    npv_b=ctx.portfolio_npv_b,
    irr_a=ctx.portfolio_irr_a,
    irr_b=ctx.portfolio_irr_b,
    total_investment=ctx.total_cost,
    total_annual_installment=ctx.total_annual_installment,
    installment_years=ctx.total_installment_years,
    discount_rate=ctx.discount_rate,
)
show.financing_recommendation(
    rec,
    title=(
        f"4) Financing Decision — "
        f"Option A €{ctx.total_cost:,.0f} upfront vs. "
        f"Option B €{ctx.total_annual_installment:,.0f}/yr over {ctx.total_installment_years} years"
    ),
)

---

## Quick Reference

| Term | Definition |
|------|------------|
| **NPV** | Net Present Value — sum of discounted future cashflows minus the initial investment. Positive NPV = value creation. |
| **IRR** | Internal Rate of Return — the discount rate at which NPV = 0. Higher IRR = faster capital recovery. |
| **PI** | Profitability Index — NPV per euro of invested capital (in PV terms). PI > 1 means the project returns more than it costs. |
| **Business Value Floor 95** | The 5th percentile of the Monte Carlo simulation. 95% of scenarios are better than this floor. |
| **P95** | The 95th percentile — the ceiling. Only 5% of scenarios exceed this value. |
| **OpEx** | Annual operating cost — recurring expense deducted from business value every year. |
| **Hurdle rate** | Minimum required return (= discount rate). Projects must clear this to create shareholder value. |
| **Monte Carlo** | Simulation technique: run many random scenarios per feature to build a probability distribution of outcomes. |

*Previous: [A01 — Portfolio Advisor](01-portfolio-advisor.ipynb)*